# Lab 03 — المخرجات المنظمة والتحقق من البيانات

**نظرة عامة:** تحويل رسائل المرضى النصية إلى بيانات منظمة باستخدام Pydantic، مع التحقق من القيم، إصلاح الأخطاء تلقائيًا، وعرض نتائج الفرز الطبي.

راجع ملف `README.md` داخل هذا المجلد للحصول على شرح مختصر ومنظم للأهداف، طريقة العمل، النتائج، وأهم النقاط التعليمية.


In [1]:
ASSISTANT_NAME = "Farah Hospital AI"

DEPARTMENTS = [
    "emergency",
    "cardiology",
    "dermatology",
    "orthopedics",
    "general_medicine",
    "other"
]

CITIES = [
    "Riyadh",
    "Jeddah",
    "Dammam",
    "Khobar",
    "Qatif",
    "unknown"
]

PRIORITIES = [
    "NORMAL",
    "URGENT",
    "CRITICAL"
]

print("Assistant:", ASSISTANT_NAME)
print("Departments:", DEPARTMENTS)
print("Cities:", CITIES)
print("Priorities:", PRIORITIES)

Assistant: Farah Hospital AI
Departments: ['emergency', 'cardiology', 'dermatology', 'orthopedics', 'general_medicine', 'other']
Cities: ['Riyadh', 'Jeddah', 'Dammam', 'Khobar', 'Qatif', 'unknown']
Priorities: ['NORMAL', 'URGENT', 'CRITICAL']


In [2]:
from pydantic import BaseModel, field_validator, ValidationError

class PatientTicket(BaseModel):
    patient_name: str
    department: str
    city: str
    priority: str
    symptom: str

    @field_validator("department")
    @classmethod
    def check_department(cls, value):
        if value not in DEPARTMENTS:
            raise ValueError("Department is not supported")
        return value

    @field_validator("city")
    @classmethod
    def check_city(cls, value):
        if value not in CITIES:
            raise ValueError("City is not supported")
        return value

    @field_validator("priority")
    @classmethod
    def check_priority(cls, value):
        if value not in PRIORITIES:
            raise ValueError("Priority must be NORMAL, URGENT, or CRITICAL")
        return value

In [3]:
ticket = PatientTicket(
    patient_name="Farah",
    department="cardiology",
    city="Riyadh",
    priority="URGENT",
    symptom="chest discomfort"
)

print(ticket)

patient_name='Farah' department='cardiology' city='Riyadh' priority='URGENT' symptom='chest discomfort'


In [4]:
try:
    ticket = PatientTicket(
        patient_name="Sara",
        department="neurology",
        city="London",
        priority="HIGH",
        symptom="headache"
    )

    print(ticket)

except ValidationError as e:
    for error in e.errors():
        print(error["loc"][0], "→", error["msg"])

department → Value error, Department is not supported
city → Value error, City is not supported
priority → Value error, Priority must be NORMAL, URGENT, or CRITICAL


In [5]:
MESSAGES = [
    "My name is Sara. I am in Riyadh and I have severe chest pain and difficulty breathing.",

    "Hi, I'm Farah from Khobar. I fell down the stairs, my leg is swollen and I cannot walk.",

    "I'm Ahmed in Dammam. I have a mild skin rash on my hand.",

    "My name is Nora from Qatif. I have fever, sore throat and fatigue.",

    "I'm Khalid in Jeddah. I suddenly feel crushing chest pain and I am struggling to breathe."
]


DEMO_ANSWERS = {

    MESSAGES[0]: [
        ("Sara", "cardiology", "Riyadh", "CRITICAL",
         "severe chest pain and difficulty breathing"),

        ("Sara", "emergency", "Riyadh", "CRITICAL",
         "severe chest pain and difficulty breathing")
    ],

    MESSAGES[1]: [
        ("Farah", "orthopedic", "Khobar", "URGENT",
         "swollen leg and unable to walk"),

        ("Farah", "orthopedics", "Khobar", "URGENT",
         "swollen leg and unable to walk")
    ],

    MESSAGES[2]: [
        ("Ahmed", "dermatology", "Dammam", "NORMAL",
         "mild skin rash")
    ],

    MESSAGES[3]: [
        ("Nora", "general_medicine", "Qatif", "NORMAL",
         "fever, sore throat and fatigue")
    ],

    MESSAGES[4]: [
        ("Khalid", "emergency", "Jeddah", "CRITICAL",
         "crushing chest pain and difficulty breathing")
    ]
}

In [6]:
def demo_model(message, errors=None):

    answers = DEMO_ANSWERS[message]

    if errors is None:
        answer = answers[0]
    else:
        answer = answers[-1]

    name, department, city, priority, symptom = answer

    return {
        "patient_name": name,
        "department": department,
        "city": city,
        "priority": priority,
        "symptom": symptom
    }


print(demo_model(MESSAGES[0]))

{'patient_name': 'Sara', 'department': 'cardiology', 'city': 'Riyadh', 'priority': 'CRITICAL', 'symptom': 'severe chest pain and difficulty breathing'}


In [7]:
def extract_ticket(message):

    errors = None

    for attempt in [1, 2]:

        raw = demo_model(message, errors)

        try:
            ticket = PatientTicket(**raw)

            if attempt == 1:
                return ticket, "first_try"

            return ticket, "after_repair"

        except ValidationError as e:

            errors = e.errors()

            print(
                "Attempt",
                attempt,
                "failed →",
                errors[0]["msg"]
            )

    return None, "escalated"

In [8]:
ticket, status = extract_ticket(MESSAGES[1])

print("Status:", status)
print("Ticket:", ticket)

Attempt 1 failed → Value error, Department is not supported
Status: after_repair
Ticket: patient_name='Farah' department='orthopedics' city='Khobar' priority='URGENT' symptom='swollen leg and unable to walk'


In [9]:
for message in MESSAGES:

    ticket, status = extract_ticket(message)

    print("\nPatient Message:")
    print(message)

    print("Status:", status)
    print("Structured Ticket:", ticket)

    print("-" * 60)


Patient Message:
My name is Sara. I am in Riyadh and I have severe chest pain and difficulty breathing.
Status: first_try
Structured Ticket: patient_name='Sara' department='cardiology' city='Riyadh' priority='CRITICAL' symptom='severe chest pain and difficulty breathing'
------------------------------------------------------------
Attempt 1 failed → Value error, Department is not supported

Patient Message:
Hi, I'm Farah from Khobar. I fell down the stairs, my leg is swollen and I cannot walk.
Status: after_repair
Structured Ticket: patient_name='Farah' department='orthopedics' city='Khobar' priority='URGENT' symptom='swollen leg and unable to walk'
------------------------------------------------------------

Patient Message:
I'm Ahmed in Dammam. I have a mild skin rash on my hand.
Status: first_try
Structured Ticket: patient_name='Ahmed' department='dermatology' city='Dammam' priority='NORMAL' symptom='mild skin rash'
------------------------------------------------------------

Pat

In [10]:
successful = 0
repaired = 0
escalated = 0

for message in MESSAGES:

    ticket, status = extract_ticket(message)

    if ticket is not None:
        successful += 1

    if status == "after_repair":
        repaired += 1

    if status == "escalated":
        escalated += 1


pass_rate = (successful / len(MESSAGES)) * 100

print("========== FARAH HOSPITAL AI REPORT ==========")
print("Total Cases:", len(MESSAGES))
print("Successful Cases:", successful)
print("Automatically Repaired:", repaired)
print("Escalated Cases:", escalated)
print("Pass Rate:", round(pass_rate, 1), "%")

Attempt 1 failed → Value error, Department is not supported
========== FARAH HOSPITAL AI REPORT ==========
Total Cases: 5
Successful Cases: 5
Automatically Repaired: 1
Escalated Cases: 0
Pass Rate: 100.0 %


In [11]:
print("=" * 70)
print("        FARAH HOSPITAL — AI TRIAGE DASHBOARD")
print("=" * 70)

for message in MESSAGES:

    ticket, status = extract_ticket(message)

    if ticket is None:
        print("⚠️ Case requires manual review")
        continue

    if ticket.priority == "CRITICAL":
        icon = "🚨"
    elif ticket.priority == "URGENT":
        icon = "⚠️"
    else:
        icon = "✅"

    print()
    print(icon, "PATIENT:", ticket.patient_name)
    print("   City:", ticket.city)
    print("   Department:", ticket.department)
    print("   Priority:", ticket.priority)
    print("   Symptom:", ticket.symptom)
    print("   Extraction:", status)

        FARAH HOSPITAL — AI TRIAGE DASHBOARD

🚨 PATIENT: Sara
   City: Riyadh
   Department: cardiology
   Priority: CRITICAL
   Symptom: severe chest pain and difficulty breathing
   Extraction: first_try
Attempt 1 failed → Value error, Department is not supported

⚠️ PATIENT: Farah
   City: Khobar
   Department: orthopedics
   Priority: URGENT
   Symptom: swollen leg and unable to walk
   Extraction: after_repair

✅ PATIENT: Ahmed
   City: Dammam
   Department: dermatology
   Priority: NORMAL
   Symptom: mild skin rash
   Extraction: first_try

✅ PATIENT: Nora
   City: Qatif
   Department: general_medicine
   Priority: NORMAL
   Symptom: fever, sore throat and fatigue
   Extraction: first_try

🚨 PATIENT: Khalid
   City: Jeddah
   Department: emergency
   Priority: CRITICAL
   Symptom: crushing chest pain and difficulty breathing
   Extraction: first_try


In [12]:
department_counts = {}

for message in MESSAGES:

    ticket, status = extract_ticket(message)

    if ticket is not None:

        department = ticket.department

        if department not in department_counts:
            department_counts[department] = 0

        department_counts[department] += 1


print("===== DEPARTMENT WORKLOAD =====")

for department, count in department_counts.items():
    print(department, "→", count, "patient(s)")

Attempt 1 failed → Value error, Department is not supported
===== DEPARTMENT WORKLOAD =====
cardiology → 1 patient(s)
orthopedics → 1 patient(s)
dermatology → 1 patient(s)
general_medicine → 1 patient(s)
emergency → 1 patient(s)


In [14]:
department_counts = {}

for message in MESSAGES:

    ticket, status = extract_ticket(message)

    if ticket is not None:

        department = ticket.department

        if department not in department_counts:
            department_counts[department] = 0

        department_counts[department] += 1


print("===== DEPARTMENT WORKLOAD =====")

for department, count in department_counts.items():
    print(department, "→", count, "patient(s)")

Attempt 1 failed → Value error, Department is not supported
===== DEPARTMENT WORKLOAD =====
cardiology → 1 patient(s)
orthopedics → 1 patient(s)
dermatology → 1 patient(s)
general_medicine → 1 patient(s)
emergency → 1 patient(s)
